# Лабораторная работа № 3. Машинный перевод и оценка качества

**Дисциплина:** Основы машинного обучения
**Направление:** Цифровая лингвистика и локализация

## Цель

Измерить качество настоящего машинного перевода автоматическими метриками,
разметить его ошибки вручную, найти места, где метрика и человек расходятся,
и построить средства, которые эти места закрывают.

## Задачи

1. Реализовать BLEU и chrF и понять, что именно они считают.
2. Применить их к реальному выходу системы перевода.
3. **Разметить ошибки** вручную по упрощённой схеме MQM.
4. Найти сегменты, где высокая метрика скрывает критическую ошибку,
   и наоборот.
5. **Построить два средства**, повышающих качество приёмки: формальные
   проверки и семантическую близость. Измерить, что ловит каждое.

## Почему эта работа — центральная в курсе

Обучать модели вы, скорее всего, не будете: их берут готовыми. А вот
**оценивать их работу** будете постоянно, и здесь у лингвиста есть
преимущество перед инженером, которое можно измерить числом.

Обучать здесь ничего не нужно: перевод уже сделан, метрики пишутся
в двадцать строк, эмбеддинги посчитаны заранее.

---
## 1. Данные: настоящий машинный перевод

Столбец `ru_mt` — **реальный выход модели `Helsinki-NLP/opus-mt-en-ru`**,
полученный прогоном всех 160 английских сегментов корпуса. Это не
придуманные ошибки: всё, что вы увидите, модель сделала сама.

Для каждого сегмента есть три текста:

* `en` — оригинал,
* `ru_ref` — **эталонный** перевод (reference), сделанный человеком,
* `ru_mt` — **проверяемый** перевод (hypothesis), выход модели.

Подробности о модели и о том, как воспроизвести прогон, — в
`data/О_данных.md`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from collections import Counter

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 60)

mt = pd.read_csv("data/loc_corpus.csv")
print("Сегментов:", len(mt))
print(mt["type"].value_counts().to_dict())
mt[["en", "ru_ref", "ru_mt"]].head(6)

### Прочитайте это глазами, прежде чем считать

Первое, что делают с выходом системы перевода, — читают его. Числа
появятся дальше; сначала составьте собственное впечатление.

In [ ]:
for _, r in mt.sample(8, random_state=3).iterrows():
    print("EN :", r["en"])
    print("REF:", r["ru_ref"])
    print("MT :", r["ru_mt"])
    print("-" * 74)

---
## 2. Почему accuracy здесь не работает

В Л.р. № 1 мы сравнивали предсказанный класс с правильным: совпало или нет.
Для перевода так нельзя. У предложения **нет одного правильного перевода**:
«Сохранить изменения» и «Сохранение изменений» оба допустимы, а точное
совпадение строк дало бы 0.

Нужна метрика, измеряющая **степень** совпадения с эталоном. И сразу же —
главное ограничение подхода: эталон один, а верных переводов много.

In [ ]:
import re

TOKEN = re.compile(r"\w+", re.UNICODE)

def tokenize(text):
    '''Слова в нижнем регистре, без пунктуации.'''
    return TOKEN.findall(str(text).lower())

print(tokenize("Не удалось подключиться к серверу"))
print(tokenize("Невозможно подключиться к серверу"))

---
## 3. BLEU: сколько n-грамм гипотезы нашлось в эталоне

**BLEU** — самая известная метрика машинного перевода:

1. для n = 1, 2, 3, 4 берётся доля n-грамм гипотезы, встретившихся в эталоне
   (это **точность**, precision);
2. четыре доли усредняются геометрически;
3. результат домножается на **штраф за краткость** (brevity penalty) —
   иначе выгодно было бы выдавать одно слово, наверняка попадающее в эталон.

Чего в этом определении нет: смысла, грамматичности, синонимов. BLEU не
знает, что «сохранить» и «сохранение» — одна лексема.

In [ ]:
def ngrams(seq, n):
    '''Счётчик n-грамм.'''
    return Counter(tuple(seq[i:i + n])
                   for i in range(len(seq) - n + 1))

def bleu(hyp, ref, max_n=4, smoothing=1.0):
    '''Упрощённый BLEU для одного предложения, значение от 0 до 1.'''
    h, r = tokenize(hyp), tokenize(ref)
    if not h or not r:
        return 0.0

    precisions = []
    for n in range(1, max_n + 1):
        hn, rn = ngrams(h, n), ngrams(r, n)
        total = sum(hn.values())
        if total == 0:                      # гипотеза короче, чем n
            continue
        # сглаживание: без него одна нулевая точность обнуляет весь BLEU
        matched = sum(min(c, rn[g]) for g, c in hn.items())
        precisions.append((matched + smoothing) / (total + smoothing))

    if not precisions:
        return 0.0
    geo_mean = np.exp(np.mean(np.log(precisions)))

    penalty = 1.0 if len(h) > len(r) else np.exp(1 - len(r) / max(len(h), 1))
    return float(geo_mean * penalty)

print(f"{bleu('Сохранить изменения', 'Сохранить изменения'):.3f}  идентично")
print(f"{bleu('Сохранение изменений', 'Сохранить изменения'):.3f}  та же лексика, другие формы")
print(f"{bleu('Удалось подключиться к серверу', 'Не удалось подключиться к серверу'):.3f}  потеряно отрицание")

> **Первое наблюдение.** Перевод с потерянным отрицанием — то есть с прямо
> противоположным смыслом — получает высокую оценку: пропало одно короткое
> слово из пяти. А корректный перевод другими словоформами получает низкую.
> BLEU считает форму, а не смысл, и для русского с его словоизменением это
> бьёт особенно сильно.

> **Про сглаживание.** Без него BLEU на коротком сегменте почти всегда равен
> нулю: в строке из трёх слов просто нет 4-грамм. Необходимость такой
> заплатки сама по себе говорит, что **BLEU плохо приспособлен к коротким
> сегментам** — то есть ровно к тому, из чего состоит локализация интерфейса.

---
## 4. chrF: то же самое, но по символам

**chrF** сравнивает символьные n-граммы и объединяет точность с полнотой
в F-меру. Для языков с богатым словоизменением это работает лучше:
«сохранить» и «сохранение» имеют общую основу, а значит общие символьные
n-граммы, даже если как слова они различны.

In [ ]:
def chrf(hyp, ref, max_n=6, beta=2.0):
    '''chrF: F-мера по символьным n-граммам, значение от 0 до 1.'''
    h = " ".join(tokenize(hyp))
    r = " ".join(tokenize(ref))
    if not h or not r:
        return 0.0

    f_scores = []
    for n in range(1, max_n + 1):
        hn, rn = ngrams(h, n), ngrams(r, n)
        if not hn or not rn:
            continue
        matched = sum(min(c, rn[g]) for g, c in hn.items())
        precision = matched / sum(hn.values())
        recall  = matched / sum(rn.values())
        if precision + recall == 0:
            f_scores.append(0.0)
        else:
            f_scores.append((1 + beta**2) * precision * recall
                            / (beta**2 * precision + recall))
    return float(np.mean(f_scores)) if f_scores else 0.0

for h, r, note in [
    ("Сохранить изменения", "Сохранить изменения", "идентично"),
    ("Сохранение изменений", "Сохранить изменения", "другие формы"),
    ("Удалось подключиться к серверу", "Не удалось подключиться к серверу", "потеряно отрицание"),
]:
    print(f"BLEU {bleu(h, r):.3f}   chrF {chrf(h, r):.3f}   {note}")

> **Второе наблюдение.** chrF заметно добрее к смене словоформы — именно то,
> что нужно для русского. Но потерю отрицания он не замечает так же, как
> BLEU: «не» — это два символа.
>
> **Ни одна метрика, сравнивающая формы, не может поймать инверсию смысла.**
> Это свойство подхода, а не недостаток реализации.

---
## 5. Считаем по всему корпусу

In [ ]:
mt["BLEU"] = [bleu(h, r) for h, r in zip(mt["ru_mt"], mt["ru_ref"])]
mt["chrF"] = [chrf(h, r) for h, r in zip(mt["ru_mt"], mt["ru_ref"])]

print(f"Средний BLEU: {mt['BLEU'].mean():.3f}   медиана: {mt['BLEU'].median():.3f}")
print(f"Средний chrF: {mt['chrF'].mean():.3f}   медиана: {mt['chrF'].median():.3f}")
print()
print(mt.groupby("type")[["BLEU", "chrF"]].mean().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, color in zip(axes, ["BLEU", "chrF"], ["#4C72B0", "#55A868"]):
    ax.hist(mt[metric], bins=16, color=color, edgecolor="white")
    ax.axvline(mt[metric].median(), color="#B03A2B", linestyle="--",
               label=f"медиана {mt[metric].median():.2f}")
    ax.set_xlabel(metric); ax.set_ylabel("число сегментов")
    ax.legend()
fig.suptitle("Распределение оценок по 160 сегментам реального перевода")
plt.tight_layout()
plt.show()

> **Обратите внимание на разброс по типам контента.** Метрики выше всего
> на документации и ниже всего на маркетинге. Это не значит, что маркетинг
> переведён хуже: он переведён **иначе**, чем эталон, потому что у
> рекламного текста больше допустимых вариантов. Метрика штрафует
> вариативность, а не ошибку.

---
## 6. Ручная разметка ошибок

Размечаем по упрощённой схеме **MQM** (Multidimensional Quality Metrics) —
отраслевому стандарту оценки перевода.

### Категории ошибок

| категория | что означает | пример |
|---|---|---|
| `точность` | смысл искажён, добавлен или потерян | пропало предложение |
| `беглость` | нарушены нормы языка перевода | сбой согласования |
| `терминология` | термин не тот, что принят в проекте | «любимые» вместо «избранное» |
| `стиль` | нарушен регистр или тон | «ты» вместо «вы» в интерфейсе |
| `разметка` | сломаны плейсхолдеры, теги, форматирование | `{name}` переведён |
| `нет` | ошибок не обнаружено | |

### Степень серьёзности

| степень | балл | когда ставить |
|---|---|---|
| `критическая` | 25 | продукт сломан или смысл противоположный |
| `серьёзная` | 5 | пользователь будет введён в заблуждение |
| `незначительная` | 1 | заметно специалисту, задачу не срывает |
| `нет` | 0 | |

Баллы — веса из MQM: одна критическая ошибка «стоит» двадцати пяти
незначительных. Они отражают **стоимость последствий**, а не частоту.

In [ ]:
# Разбор десяти реальных сегментов. Так должна выглядеть ваша таблица
# в задании 3 — только длиннее.
example = pd.DataFrame([
    {"id": "s002", "категория": "разметка", "серьёзность": "критическая",
     "комментарий": "плейсхолдер {name} превращён в \"(имя}\" — строка не соберётся"},
    {"id": "s012", "категория": "разметка", "серьёзность": "критическая",
     "комментарий": "плейсхолдер {count} переведён как {подсчитать}"},
    {"id": "s028", "категория": "разметка", "серьёзность": "критическая",
     "комментарий": "плейсхолдер {minutes} переведён как {минуты}"},
    {"id": "s083", "категория": "точность", "серьёзность": "критическая",
     "комментарий": "второе предложение потеряно целиком"},
    {"id": "s115", "категория": "точность", "серьёзность": "критическая",
     "комментарий": "потеряна половина сегмента, plan передано как «план»"},
    {"id": "s003", "категория": "точность", "серьёзность": "серьёзная",
     "комментарий": "account передано как «счёт» вместо «учётная запись»"},
    {"id": "s026", "категория": "терминология", "серьёзность": "серьёзная",
     "комментарий": "«Возвращение из резервного копирования» вместо «Восстановить из резервной копии»"},
    {"id": "s114", "категория": "стиль", "серьёзность": "серьёзная",
     "комментарий": "обращение на «ты» в маркетинге делового продукта"},
    {"id": "s006", "категория": "терминология", "серьёзность": "незначительная",
     "комментарий": "«любимые» вместо принятого в интерфейсах «избранное»"},
    {"id": "s065", "категория": "нет", "серьёзность": "нет",
     "комментарий": "корректная перефразировка при очень низком BLEU"},
])

PENALTY = {"критическая": 25, "серьёзная": 5, "незначительная": 1, "нет": 0}
example["штраф"] = example["серьёзность"].map(PENALTY)

review = example.merge(mt[["id", "en", "ru_ref", "ru_mt", "BLEU", "chrF"]], on="id")
review[["id", "ru_ref", "ru_mt", "серьёзность", "штраф", "BLEU", "chrF"]].round(3)

---
## 7. Главный результат: где метрика расходится с человеком

In [ ]:
median_bleu = mt["BLEU"].median()

for _, r in review.sort_values("штраф", ascending=False).iterrows():
    flag = ""
    if r["штраф"] >= 25 and r["BLEU"] > median_bleu:
        flag = "   <<< МЕТРИКА НЕ ЗАМЕТИЛА КРИТИЧЕСКУЮ ОШИБКУ"
    if r["штраф"] == 0 and r["BLEU"] < median_bleu:
        flag = "   <<< МЕТРИКА НАКАЗАЛА КОРРЕКТНЫЙ ПЕРЕВОД"
    print("=" * 78)
    print(f"[{r['id']}] BLEU {r['BLEU']:.3f}  chrF {r['chrF']:.3f}  "
          f"штраф {r['штраф']}{flag}")
    print("  оригинал:", r["en"])
    print("  эталон  :", r["ru_ref"])
    print("  перевод :", r["ru_mt"])
    print("  разбор  :", r["комментарий"])

> **Расхождение работает в обе стороны, и это реальные сегменты.**
>
> * **s002** — плейсхолдер `{name}` превращён в `"(имя}"`. Строка интерфейса
>   не соберётся, пользователь увидит мусор. BLEU **выше медианы**: для
>   метрики это одно непопавшее слово из трёх.
> * **s065** — «Приведенный ниже пример показывает минимальную рабочую
>   конфигурацию» вместо «В следующем примере показана…». Перевод
>   безупречен. BLEU **ниже медианы**, потому что переставлены слова
>   и выбраны синонимы.
>
> Приёмка по одному числу отклонила бы верный перевод и пропустила
> сломанную строку.

In [ ]:
# Согласие разметчиков: разметка тоже измерение, и у неё есть надёжность.
from sklearn.metrics import cohen_kappa_score

annotator_1 = ["разметка", "разметка", "разметка", "точность", "точность",
                "точность",     "терминология", "стиль", "терминология", "нет"]
annotator_2 = ["разметка", "разметка", "разметка", "точность", "точность",
                "терминология", "терминология", "стиль", "стиль",        "нет"]

k = cohen_kappa_score(annotator_1, annotator_2)
print(f"Каппа Коэна: {k:.3f}")
print("Расхождение:", [(a, b) for a, b in zip(annotator_1, annotator_2) if a != b])

| каппа | как читать |
|---|---|
| < 0.20 | согласия практически нет |
| 0.21-0.40 | слабое |
| 0.41-0.60 | умеренное |
| 0.61-0.80 | существенное |
| > 0.80 | почти полное |

> **Расхождение содержательное, а не небрежность.** `account` → «счёт» —
> это ошибка точности (смысл другой) или терминологии (не тот термин)?
> Оба ответа защитимы, пока руководство для разметчиков не решило иначе.
> Написать это руководство — профессиональная задача лингвиста: модель не
> может решить, что считать ошибкой, она может лишь применить уже принятое
> решение.

---
## 8. Средства, которые повысят качество приёмки

Мы установили, что метрика ошибается в обе стороны. Теперь построим два
средства, закрывающих эти дыры, и **измерим, что ловит каждое**.

### Средство первое: формальные проверки

То, что делают инструменты контроля качества перевода (Xbench, Verifika,
QA Distiller). Идея простая: часть требований к переводу проверяется
механически, без всякого понимания смысла.

* плейсхолдеры в оригинале и переводе должны совпадать;
* числа должны совпадать;
* если в оригинале есть отрицание, оно должно быть и в переводе;
* перевод не должен быть пустым или подозрительно коротким.

In [ ]:
PLACEHOLDER = re.compile(r"\{[^}]*\}|%[sd]|<[^>]+>")
NUMBER      = re.compile(r"\d+")
NEG_EN      = re.compile(r"\b(not|no|never|cannot|can't|don't|doesn't|unable|without|nor)\b", re.I)
NEG_RU      = re.compile(r"\b(не|нет|ни|нельзя|без|никогда|никаких|отсутствует|запрещ\w*)\b", re.I)

def formal_checks(en, ru):
    '''Возвращает список сработавших проверок. Пустой список — нареканий нет.'''
    en, ru = str(en), str(ru)
    fired = []
    if set(PLACEHOLDER.findall(en)) != set(PLACEHOLDER.findall(ru)):
        fired.append("плейсхолдеры")
    if sorted(NUMBER.findall(en)) != sorted(NUMBER.findall(ru)):
        fired.append("числа")
    if NEG_EN.search(en) and not NEG_RU.search(ru):
        fired.append("потеряно отрицание")
    if not ru.strip():
        fired.append("пусто")
    if len(ru) < 0.45 * len(en):
        fired.append("подозрительно коротко")
    return fired

mt["проверки"] = [formal_checks(e, r) for e, r in zip(mt["en"], mt["ru_mt"])]
mt["есть_замечания"] = mt["проверки"].apply(bool)

print(f"Проверки сработали на {mt['есть_замечания'].sum()} сегментах из {len(mt)}")
print(Counter(x for p in mt["проверки"] for x in p))

In [ ]:
for _, r in mt[mt["есть_замечания"]].iterrows():
    print(f"[{r['id']}] {', '.join(r['проверки'])}   (BLEU {r['BLEU']:.2f})")
    print("   EN :", r["en"])
    print("   REF:", r["ru_ref"])
    print("   MT :", r["ru_mt"])

> **Восемь сегментов из ста шестидесяти — и разбирать их надо все.**
>
> Пять срабатываний — настоящие дефекты: три сломанных плейсхолдера
> (s002, s012, s028), потерянное предложение (s083) и потерянная половина
> сегмента (s115).
>
> Три — **ложные тревоги**, и каждая учит чему-то о проектировании проверок:
>
> * **s010** «Невозможно подключиться» — отрицание выражено словом,
>   которого нет в моём списке. Проверка на отрицание требует
>   поддерживаемого словаря, а не пяти частиц.
> * **s031** «Результаты отсутствуют» — то же самое: `отсутствуют` не
>   попало под шаблон `отсутствует`.
> * **s148** «eighteen» → «18» — законная нормализация числительного,
>   но проверка чисел этого не знает.
>
> **Ложная тревога дешевле пропуска.** Разобрать восемь сегментов — минуты;
> сломанная строка в релизе стоит дороже. Именно поэтому такие проверки
> настраивают на высокую полноту, сознательно принимая ложные срабатывания.

### Средство второе: семантическая близость

Формальные проверки не знают смысла. Метрики не знают синонимов. Третий
инструмент — **готовые многоязычные эмбеддинги**: векторы, обученные на
огромных корпусах так, что близкие по смыслу тексты оказываются рядом.

Считать их здесь не нужно — они посчитаны заранее и лежат в `data/`.
Модель и способ расчёта описаны в `data/О_данных.md`. Так устроены и
современные метрики вроде COMET, только они ещё дообучены на человеческих
оценках.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

E_ref = np.load("data/emb_ru_ref.npy")
E_mt  = np.load("data/emb_ru_mt.npy")
print("Эмбеддинги:", E_ref.shape, "— по одному вектору на сегмент")

# косинус между эталоном и переводом одного и того же сегмента
mt["семантика"] = [float(E_ref[i] @ E_mt[i]) for i in range(len(mt))]

print(f"средняя семантическая близость: {mt['семантика'].mean():.3f}")
review2 = review.merge(mt[["id", "семантика", "проверки"]], on="id")
review2[["id", "серьёзность", "штраф", "BLEU", "chrF", "семантика"]].round(3)

> **Смотрите на строку s065.** BLEU 0.16 — метрика посчитала корректный
> перевод плохим. Семантическая близость 0.99 — ложная тревога снята.
> Это и есть то, ради чего берут семантические метрики.
>
> И смотрите на s002. Семантическая близость 0.80: сломанный плейсхолдер
> **не пойман и здесь**. Смысл-то передан — сломана форма.

### Сводка: какое средство что ловит

Проверим на всех размеченных сегментах, кто из трёх инструментов
поднимает тревогу там, где она нужна.

In [ ]:
summary = review2.copy()
summary["BLEU ниже медианы"]   = summary["BLEU"] < median_bleu
summary["семантика < 0.7"]     = summary["семантика"] < 0.70
summary["проверка сработала"]  = summary["проверки"].apply(bool)
summary["нужна тревога"]       = summary["штраф"] > 0

flags = summary[["id", "штраф", "нужна тревога", "BLEU ниже медианы",
                 "семантика < 0.7", "проверка сработала"]]
print(flags.to_string(index=False))

needed = summary["нужна тревога"]
print()
for name in ["BLEU ниже медианы", "семантика < 0.7", "проверка сработала"]:
    caught       = (summary[name] & needed).sum()
    false_alarms = (summary[name] & ~needed).sum()
    print(f"{name:22s} поймано {caught} из {needed.sum()},  ложных тревог {false_alarms}")

### Цена тревоги

Полнота — половина дела. Вторая половина: **сколько сегментов средство
отправляет на ручную проверку.** Средство, помечающее половину корпуса,
формально «ловит много», но экономит ноль времени.

In [ ]:
total = len(mt)
mt["семантика"] = [float(E_ref[i] @ E_mt[i]) for i in range(len(mt))]

cost = pd.DataFrame([
    {"средство": "BLEU ниже медианы",
     "на проверку": int((mt["BLEU"] < median_bleu).sum())},
    {"средство": "chrF ниже медианы",
     "на проверку": int((mt["chrF"] < mt["chrF"].median()).sum())},
    {"средство": "семантика < 0.7",
     "на проверку": int((mt["семантика"] < 0.70).sum())},
    {"средство": "формальные проверки",
     "на проверку": int(mt["есть_замечания"].sum())},
])
cost["доля корпуса"] = (cost["на проверку"] / total * 100).round(1).astype(str) + " %"
print(f"Всего сегментов: {total}")
print(cost.to_string(index=False))

> **Вот почему сравнение по одной полноте обманывает.** Порог «BLEU ниже
> медианы» по построению помечает половину корпуса — около восьмидесяти
> сегментов. Формальные проверки помечают восемь, то есть в десять раз
> меньше работы для редактора.
>
> И решающий случай: сегмент **s002** со сломанным плейсхолдером нашли
> **только** проверки. Его BLEU выше медианы, в отбор по метрике он не
> попал бы вообще — а это строка, которая не соберётся в продукте.
>
> Средство ценно не тем, сколько оно ловит, а отношением найденного
> к просмотренному.

> **Вывод, ради которого делалась вся работа.**
>
> Ни одно средство не закрывает задачу в одиночку:
>
> | средство | ловит | не ловит |
> |---|---|---|
> | BLEU / chrF | грубые расхождения с эталоном | инверсию смысла, поломки разметки; штрафует синонимию |
> | формальные проверки | плейсхолдеры, числа, пропуски, отрицание | всё, что требует понимания |
> | семантическая близость | искажение смысла, снимает ложные тревоги метрик | поломки формы при сохранённом смысле |
>
> **Приёмка строится из нескольких средств, а не из одного числа.**
> Формальные проверки ставят первыми — они дешёвые и с полной полнотой
> на своём классе ошибок. Метрики годятся для сравнения систем на большом
> корпусе. Смысловые решения остаются человеку, и именно поэтому
> разметка из раздела 6 — не учебное упражнение, а описание работы.

---
## 9. Сверка с эталонной реализацией метрик (необязательный раздел)

Наши BLEU и chrF — учебные упрощения. Отраслевой стандарт — библиотека
`sacrebleu`. Если она установлена, сравните значения.

In [ ]:
try:
    import sacrebleu
    ref_bleu = sacrebleu.corpus_bleu(
        mt["ru_mt"].tolist(), [mt["ru_ref"].tolist()]).score / 100
    ref_chrf = sacrebleu.corpus_chrf(
        mt["ru_mt"].tolist(), [mt["ru_ref"].tolist()]).score / 100
    print(f"наш BLEU (среднее по сегментам): {mt['BLEU'].mean():.3f}")
    print(f"sacrebleu BLEU (по корпусу)    : {ref_bleu:.3f}")
    print(f"наш chrF (среднее по сегментам): {mt['chrF'].mean():.3f}")
    print(f"sacrebleu chrF (по корпусу)    : {ref_chrf:.3f}")
    print()
    print("Значения не обязаны совпадать: BLEU по корпусу считается не как")
    print("среднее BLEU по предложениям, и сглаживание у нас своё.")
    print("Объясните разницу в отчёте — расхождение по chrF заметно меньше,")
    print("и это тоже требует объяснения.")
except ImportError:
    print("Библиотека sacrebleu не установлена: pip install sacrebleu==2.6.0")
    print("Раздел необязательный.")

---
## 10. Как перевод был получен (необязательный раздел)

Ячейка ниже воспроизводит прогон модели. Выполнять её не требуется:
результат уже лежит в корпусе. Она приведена, чтобы было видно, что
`ru_mt` не взялся ниоткуда, и чтобы вы могли прогнать свои сегменты.

> **Важно для Windows.** `torch` нужно импортировать **до** `numpy`,
> `pandas`, `sklearn` и `scipy`. При обратном порядке импорт падает с
> `OSError: [WinError 1114]` — это известный конфликт загрузки библиотек,
> а не ошибка в коде. В этом ноутбуке pandas уже импортирован выше,
> поэтому ячейку надёжнее выполнять в **отдельном** файле.

In [ ]:
RUN_CODE = '''
import torch                                    # ПЕРВЫМ, до pandas и numpy
from transformers import MarianMTModel, MarianTokenizer

MODEL = "Helsinki-NLP/opus-mt-en-ru"
tok = MarianTokenizer.from_pretrained(MODEL)
model = MarianMTModel.from_pretrained(MODEL).eval()

segments = ["Save changes", "Delete file \\"{name}\\"?", "No results found"]
with torch.no_grad():
    inputs = tok(segments, return_tensors="pt", padding=True)
    output = model.generate(**inputs, max_new_tokens=128, num_beams=4)
print(tok.batch_decode(output, skip_special_tokens=True))
'''
print(RUN_CODE)
print("Зависимости: pip install transformers==4.57.1 torch sentencepiece sacremoses")

---
## Задание

1. **Изучите ноутбук** и выполните его сверху вниз без ошибок.
   Необязательные разделы 9-10 могут не отработать — это нормально.

2. **Проверьте метрики на своих примерах.** Придумайте три пары
   «эталон — перевод»: (а) верный перевод другими словами, (б) грамматически
   ломаный с той же лексикой, (в) с противоположным смыслом при минимальной
   правке. Посчитайте BLEU, chrF и семантическую близость. Какая пара
   получила самую высокую оценку и почему это плохая новость?

3. **Разметьте 60 сегментов** (`s001`-`s060`) по схеме раздела 6: категория,
   серьёзность, комментарий. Сохраните в CSV и приложите к отчёту.
   Это основная часть работы.

4. **Постройте сводку** по вашей разметке: сколько ошибок каждой категории,
   какой суммарный штраф по каждому типу контента. В каком типе контента
   машинный перевод опаснее всего и почему?

5. **Расхождение метрики и человека.** Постройте диаграмму рассеяния
   «BLEU по горизонтали, штрафной балл по вертикали» для ваших 60 сегментов
   и посчитайте ранговую корреляцию. Найдите и разберите:
   * минимум два сегмента с BLEU выше медианы и серьёзной ошибкой;
   * минимум два сегмента с низким BLEU и отсутствием ошибок.

6. **Улучшите формальные проверки.** В разделе 8 три ложных срабатывания
   из восьми. Исправьте проверки так, чтобы ложных стало меньше, **не
   потеряв ни одного настоящего дефекта**. Приведите новую версию функции
   и таблицу «было / стало». Объясните, почему нельзя просто убрать
   проверку чисел.

7. **Сравните три средства на своих 60 сегментах.** Для каждого посчитайте,
   сколько ошибок со штрафом выше нуля оно поймало и сколько дало ложных
   тревог. Постройте таблицу как в разделе 8. Какое сочетание средств вы
   предложили бы для приёмки локализации интерфейса?

8. **Согласие разметчиков.** Обменяйтесь разметкой с одногруппником
   (не менее 20 общих сегментов) и посчитайте каппу Коэна отдельно по
   категории и по серьёзности. Где согласие ниже? Разберите три случая
   расхождения и сформулируйте правило, которое бы их устранило. Это
   правило — заготовка руководства для разметчиков.

9. **Вывод.** Ответьте на вопрос заказчика: «можно ли принимать перевод
   в работу, если средний BLEU выше 0.6?» Ответ обоснуйте числами из
   заданий 4-8, а не общими соображениями.

### Требования к отчёту

* ноутбук выполняется сверху вниз на чистом окружении;
* файл с разметкой приложен;
* в задании 5 приведены конкретные сегменты с разбором, а не только график;
* в задании 6 показано, что настоящие дефекты не потерялись;
* в выводе назван хотя бы один случай, когда метрика вводит в заблуждение,
  и указана цена этой ошибки для проекта локализации.